# CISC3024 Pattern Recognition — AI Assignment #2
## `00` — Colab environment setup (VBLL)

**Algorithm:** VBLL — Variational Bayesian Last Layers (Harrison, Willes & Snoek, **ICLR 2024**)

| | |
|---|---|
| Course | CISC3024 Pattern Recognition |
| Assignment | AI Assignment #2 |
| Target runtime | Google Colab, free tier, NVIDIA Tesla T4 |
| Purpose | Install and verify every dependency **before** any GPU time is spent |

> Every line of code in this notebook was generated with AI assistance, as the
> assignment brief requires. No line was written by hand.

### Why this notebook exists

Colab already ships a CUDA-matched PyTorch. The single most common way to break a
Colab session is to let a `pip install` drag in a different PyTorch build -- after
which `torch.cuda.is_available()` silently returns `False`.

This notebook therefore does **not** install PyTorch. It first records the
`torch` / `torchvision` versions Colab already has, writes them into a
**constraints file**, and installs everything else behind that constraint. No
transitive dependency can then replace them.


---
## How to run this notebook

1. `Runtime → Change runtime type → Hardware accelerator: T4 GPU` → **Save**
2. `Runtime → Run all` (or run the cells in order)

| Cell | What it does | Expected time |
|---|---|---|
| **Cell 1** | Probes the runtime, writes the constraints file, installs 13 packages | ~60–90 s |
| **Cell 2** | Verifies every import and smoke-tests the VBLL classifier API | ~10 s |
| **Cell 3** | Freezes the exact versions actually installed (`environment.json`, `requirements.lock.txt`) | ~5 s |
| **Cell 4** | CUDA diagnosis — **only run this if something is wrong** | instant |

After Cell 2 prints `ALL CHECKS PASSED`, this notebook has done its job and the
experiment notebook can be opened in the same runtime.


In [1]:
# =====================================================================
# Cell 1 -- probe the runtime, then install every dependency
# =====================================================================
# The constraints file is the important part, not the package list.
# Colab already ships a CUDA-matched torch; pinning it in a constraints
# file means no transitive dependency can swap it for a CPU wheel.
# =====================================================================
import pathlib, platform, subprocess, sys
import torch, torchvision

NL = chr(10)


def base(v):
    """'2.6.0+cu124' -> '2.6.0'  (strip the local build tag)"""
    return v.split("+")[0]


def rule(title):
    print("=" * 64)
    print(title)
    print("=" * 64)


rule("BEFORE INSTALL")
print("Python       :", platform.python_version())
print("torch        :", torch.__version__)
print("torchvision  :", torchvision.__version__)
print("CUDA runtime :", torch.version.cuda)
print("GPU          :",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
if not torch.cuda.is_available():
    print()
    print("!! No GPU detected.")
    print("   Runtime > Change runtime type > T4 GPU, then re-run this cell.")

# --- 1. pin torch / torchvision to exactly what Colab already has ----------
CONSTRAINTS = pathlib.Path("colab-constraints.txt")
CONSTRAINTS.write_text(NL.join([
    "torch==" + base(torch.__version__),
    "torchvision==" + base(torchvision.__version__),
]) + NL)

print()
rule("colab-constraints.txt  (prevents torch from being replaced)")
print(CONSTRAINTS.read_text().rstrip())

# --- 2. everything else, behind that constraint ---------------------------
PACKAGES = [
    "vbll==0.4.9",           # the algorithm under study (ICLR 2024)
    "laplace-torch==0.2.3",  # comparison baseline (last-layer Laplace)
    "numpy", "scipy", "pandas", "scikit-learn",
    "matplotlib", "seaborn", "tqdm",
]

cmd = [sys.executable, "-m", "pip", "install", "-q",
       "-c", str(CONSTRAINTS), *PACKAGES]
print()
print("$ " + " ".join(cmd))
print()
subprocess.run(cmd, check=True)

print()
rule("AFTER INSTALL")
print("torch        :", torch.__version__, "  <- must equal the BEFORE value")
print("CUDA runtime :", torch.version.cuda)
print("GPU          :",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
print()
print("Install complete -> run Cell 2 to verify.")


BEFORE INSTALL
Python       : 3.13.15
torch        : 2.11.0+cu130
torchvision  : 0.26.0+cu130
CUDA runtime : 13.0
GPU          : Tesla T4

colab-constraints.txt  (prevents torch from being replaced)
torch==2.11.0
torchvision==0.26.0

$ /usr/bin/python3 -m pip install -q -c colab-constraints.txt vbll==0.4.9 laplace-torch==0.2.3 numpy scipy pandas scikit-learn matplotlib seaborn tqdm


AFTER INSTALL
torch        : 2.11.0+cu130   <- must equal the BEFORE value
CUDA runtime : 13.0
GPU          : Tesla T4

Install complete -> run Cell 2 to verify.


In [2]:
# =====================================================================
# Cell 2 -- verification. Must end with "ALL CHECKS PASSED".
# =====================================================================
import importlib
import torch

MODULES = ["torch", "numpy", "scipy", "pandas", "sklearn",
           "matplotlib", "seaborn", "tqdm", "vbll", "laplace"]

print("%-14s%-16s%s" % ("module", "version", "status"))
print("-" * 44)
failed = []
for name in MODULES:
    try:
        m = importlib.import_module(name)
        ver = getattr(m, "__version__", "n/a")
        print("%-14s%-16s%s" % (name, ver, "OK"))
    except Exception as e:
        failed.append(name)
        print("%-14s%-16sFAIL: %s: %s" % (name, "-", type(e).__name__, e))

if failed:
    raise RuntimeError("import failed for: %s -- see the troubleshooting cell" % failed)

# --- smoke-test the exact VBLL API the experiments will use ---------------
import vbll

try:                                     # top-level API (as in the official tutorial)
    DiscClassification = vbll.DiscClassification
    GenClassification = vbll.GenClassification
except AttributeError:                   # fall back to the documented module path
    from vbll.layers.classification import DiscClassification, GenClassification

torch.manual_seed(0)
head = DiscClassification(16, 3, 1e-4, parameterization="diagonal", return_ood=True)
out = head(torch.randn(8, 16))
loss = out.train_loss_fn(torch.randint(0, 3, (8,)))

print()
print("VBLL API smoke test")
print("  DiscClassification :", type(head).__name__)
print("  train_loss_fn      :", round(float(loss), 4))
print("  predictive.probs   :", tuple(out.predictive.probs.shape))
print("  ood_scores         :", tuple(out.ood_scores.shape))
print("  GenClassification  :", GenClassification.__name__, "(importable)")

assert torch.cuda.is_available(), "CUDA unavailable -- run Cell 4 to diagnose"
print()
print("=" * 64)
print("GPU :", torch.cuda.get_device_name(0),
      "| VRAM %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))
print("ALL CHECKS PASSED")
print("=" * 64)


module        version         status
--------------------------------------------
torch         2.11.0+cu130    OK
numpy         2.1.3           OK
scipy         1.16.3          OK
pandas        2.2.3           OK
sklearn       1.6.1           OK
matplotlib    3.10.0          OK
seaborn       0.13.2          OK
tqdm          4.67.3          OK
vbll          n/a             OK
laplace       n/a             OK

VBLL API smoke test
  DiscClassification : DiscClassification
  train_loss_fn      : 1.9074
  predictive.probs   : (8, 3)
  ood_scores         : (8,)
  GenClassification  : GenClassification (importable)

GPU : Tesla T4 | VRAM 15.6 GB
ALL CHECKS PASSED


/tmp/ipykernel_392/1391204172.py:42: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  print("  train_loss_fn      :", round(float(loss), 4))


In [3]:
# =====================================================================
# Cell 3 -- freeze the environment actually used
# Produces the numbers for the report's section 10 (Reproducibility).
# =====================================================================
import json, pathlib, platform, subprocess, sys
from importlib.metadata import version, PackageNotFoundError
import torch

try:
    import torchvision
    tv = torchvision.__version__
except Exception:
    tv = None


def v(pkg):
    try:
        return version(pkg)
    except PackageNotFoundError:
        return None


TRACKED = ["vbll", "laplace-torch", "asdfghjkl", "backpack-for-pytorch",
           "curvlinops-for-pytorch", "einops", "einconv", "unfoldNd",
           "linear_operator", "opt_einsum", "torchmetrics",
           "numpy", "scipy", "pandas", "scikit-learn", "matplotlib",
           "seaborn", "tqdm"]

env = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "torch": torch.__version__,
    "torchvision": tv,
    "cuda_runtime": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "vram_gb": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
               if torch.cuda.is_available() else None,
    "packages": {p: v(p) for p in TRACKED},
}

pathlib.Path("environment.json").write_text(json.dumps(env, indent=2))

lock = subprocess.run([sys.executable, "-m", "pip", "freeze"],
                      capture_output=True, text=True).stdout
pathlib.Path("requirements.lock.txt").write_text(lock)

print(json.dumps(env, indent=2))
print()
print("wrote environment.json (%d bytes)" % len(json.dumps(env)))
print("wrote requirements.lock.txt (%d packages)" % len(lock.splitlines()))
print()
print("Download both from the Colab file browser -- they go into the report's")
print("reproducibility section and into the GitHub repository.")


{
  "python": "3.13.15",
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.39",
  "torch": "2.11.0+cu130",
  "torchvision": "0.26.0+cu130",
  "cuda_runtime": "13.0",
  "cudnn": 92700,
  "gpu": "Tesla T4",
  "vram_gb": 15.6,
  "packages": {
    "vbll": "0.4.9",
    "laplace-torch": "0.2.3",
    "asdfghjkl": "0.1a4",
    "backpack-for-pytorch": "1.7.1",
    "curvlinops-for-pytorch": "3.0.1",
    "einops": "0.8.2",
    "einconv": "0.1.0",
    "unfoldNd": "0.2.3",
    "linear_operator": "0.6.1",
    "opt_einsum": "3.4.0",
    "torchmetrics": "1.9.0",
    "numpy": "2.1.3",
    "scipy": "1.16.3",
    "pandas": "2.2.3",
    "scikit-learn": "1.6.1",
    "matplotlib": "3.10.0",
    "seaborn": "0.13.2",
    "tqdm": "4.67.3"
  }
}

wrote environment.json (632 bytes)
wrote requirements.lock.txt (732 packages)

Download both from the Colab file browser -- they go into the report's
reproducibility section and into the GitHub repository.


---
## If something breaks — CUDA version conflicts

**Golden rule: Colab's PyTorch is already correct. Never reinstall it.**

`torch.version.cuda` is the CUDA version PyTorch was *built against*; `nvidia-smi`
reports the maximum CUDA version the *driver* supports. On a fresh Colab runtime
they match. Problems appear only after a manual `pip install torch`.

| Symptom | Cause | Fix |
|---|---|---|
| `torch.cuda.is_available()` becomes `False` after installing | a dependency pulled a CPU-only torch wheel | **`Runtime → Restart session`**, then re-run Cell 1 + Cell 2. Restarting is the clean fix — no CUDA version to guess |
| `ImportError: libcudart.so.12: cannot open shared object file` | torch's CUDA runtime is newer than the driver | restart the runtime first; only if that fails, force-reinstall a wheel matching `!nvidia-smi` |
| `ImportError: undefined symbol: ...c10_cuda...` / `libc10_cuda.so` | `torch` and `torchvision` come from different builds | restart, then reinstall **both together** from the official index (see below) |
| `ERROR: Cannot install ... conflicting dependencies` | the constraints file blocked a dependency from upgrading torch — **this is the constraint working correctly** | check which package; usually an optional curvature backend. Accepting the torch bump means accepting the CUDA risk |
| `ModuleNotFoundError: No module named 'laplace'` right after a successful install | Colab's module cache is stale because torch/numpy-adjacent wheels were touched | `Runtime → Restart session`, then run **Cell 2 only** — do **not** re-run the install |
| `RuntimeError: Found no NVIDIA driver on your system` | runtime type is CPU-only | `Runtime → Change runtime type → T4 GPU` |
| `ImportError: numpy.core.multiarray failed to import` / `_ARRAY_API not found` | a package was compiled against numpy 1.x while Colab has numpy 2.x | `!pip install --upgrade <package>`, or pin `numpy<2`. This setup does not upgrade numpy, so it should not fire |
| `asdfghjkl` / `backpack-for-pytorch` fails to build | pip refused the pre-release pin | add `--pre` to the install command. The pin `asdfghjkl==0.1a4` names the pre-release explicitly, so pip normally accepts it |

### Last-resort recovery

Only if a runtime restart does not fix it. Replace `cu121` with the CUDA version
printed by `!nvidia-smi`:

```python
!pip install --force-reinstall torch torchvision --index-url https://download.pytorch.org/whl/cu121
```


In [4]:
# =====================================================================
# Cell 4 -- CUDA diagnosis. Only needed if Cell 2 failed.
# =====================================================================
import torch, subprocess

print("=" * 64)
print("nvidia-smi (driver + max CUDA the driver supports)")
print("=" * 64)
try:
    r = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
    print(r.stdout[:900] if r.stdout else r.stderr[:400])
except FileNotFoundError:
    print("nvidia-smi not found -- runtime has no GPU attached")

print()
print("=" * 64)
print("torch view")
print("=" * 64)
print("torch                  :", torch.__version__)
print("built for CUDA         :", torch.version.cuda)
print("cudnn                  :", torch.backends.cudnn.version())
print("cuda available         :", torch.cuda.is_available())
print("device count           :", torch.cuda.device_count())
print("torch is a CPU build   :", torch.version.cuda is None)

print()
print("=" * 64)
print("verdict")
print("=" * 64)
if torch.version.cuda is None:
    print("torch is a CPU-ONLY build -> a pip dependency replaced it.")
    print("FIX: Runtime > Restart session, then re-run Cell 1 (constraints protect torch).")
elif not torch.cuda.is_available():
    print("torch is a CUDA build but no device is visible -> driver/runtime mismatch.")
    print("FIX: Runtime > Restart session, then re-run Cell 1 + Cell 2.")
else:
    print("OK -- CUDA build present and device visible. Nothing to fix.")


nvidia-smi (driver + max CUDA the driver supports)
Thu Oct  8 09:54:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   63C    P8             12W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                       

torch view
torch                  : 2.11.0+cu13

---
## Next step

Once **Cell 2 prints `ALL CHECKS PASSED`**, the environment is done and this
notebook has served its purpose.

1. Keep this runtime alive — open the experiment notebook **in the same session**
   so the installed packages are still available.
2. Download `environment.json` and `requirements.lock.txt` (Cell 3). They go into
   the report's reproducibility section and into the GitHub repository.
3. The experiment notebook (`01_vbll_classification.ipynb`) is generated next.

### Verified package set

Resolved with `pip install --dry-run` on 2026-10-08 — 13 new packages, and
**`torch` / `torchvision` do not appear in the list**, i.e. Colab's CUDA build is
left untouched:

```text
asdfghjkl-0.1a4                backpack-for-pytorch-1.7.1
curvlinops-for-pytorch-3.0.1   einconv-0.1.0
einops-0.8.2                   laplace-torch-0.2.3
lightning-utilities-0.15.3     linear_operator-0.6.1
opt_einsum-3.4.0               seaborn-0.13.2
torchmetrics-1.9.0             unfoldNd-0.2.3
vbll-0.4.9
```
